### RAG pipeline - Data Ingetion to Vetor DataBase

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\saige\AppData\Local\Temp\ipykernel_21008\4064311150.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader


In [2]:
### Read all the pdf's inside the directory

def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")

        except Exception as e:
            print(f"  ✗ Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents


# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 2 PDF files to process

Processing: Association Rules using the Apriori algorithm.pdf
  ✓ Loaded 11 pages

Processing: Self Introduction.pdf
  ✓ Loaded 1 pages

Total documents loaded: 12


In [3]:
all_pdf_documents 

[Document(metadata={'producer': 'Skia/PDF m157 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Association Rules using the Apriori algorithm', 'source': '..\\data\\pdf_files\\Association Rules using the Apriori algorithm.pdf', 'total_pages': 11, 'page': 0, 'page_label': '1', 'source_file': 'Association Rules using the Apriori algorithm.pdf', 'file_type': 'pdf'}, page_content='Write\n \na\n \nprogram\n \nto\n \ngenerate\n \nAssociation\n \nRules\n \nusing\n \nthe\n \nApriori\n \nalgorithm.\n \n \nTopics\n \nyou\n \nneed\n \nto\n \nlearn\n \n1.\n \nAssociation\n \nRule\n \nMining\n \nIt\n \ndiscovers\n \nrelationships\n \nbetween\n \nitems\n \nin\n \na\n \ndataset.\n \nExample:\n \nCustomers\n \nwho\n \nbuy\n \nbread\n \nmay\n \nalso\n \nbuy\n \nbutter.\n \n \n2.\n \nTransaction\n \nDataset\n \nData\n \nis\n \nstored\n \nas\n \ntransactions\n \ncontaining\n \nitems.\n \nTransactin\n \nItems\n \nT1\n \nBread,\n \nMilk\n \nT2\n \nBread,\n \nButter\n \nT3\n \nBread,

In [4]:
### Text splitting get into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [5]:
chunks = split_documents(all_pdf_documents)
chunks

Split 12 documents into 14 chunks

Example chunk:
Content: Write
 
a
 
program
 
to
 
generate
 
Association
 
Rules
 
using
 
the
 
Apriori
 
algorithm.
 
 
Topics
 
you
 
need
 
to
 
learn
 
1.
 
Association
 
Rule
 
Mining
 
It
 
discovers
 
relationships
...
Metadata: {'producer': 'Skia/PDF m157 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Association Rules using the Apriori algorithm', 'source': '..\\data\\pdf_files\\Association Rules using the Apriori algorithm.pdf', 'total_pages': 11, 'page': 0, 'page_label': '1', 'source_file': 'Association Rules using the Apriori algorithm.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Skia/PDF m157 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Association Rules using the Apriori algorithm', 'source': '..\\data\\pdf_files\\Association Rules using the Apriori algorithm.pdf', 'total_pages': 11, 'page': 0, 'page_label': '1', 'source_file': 'Association Rules using the Apriori algorithm.pdf', 'file_type': 'pdf'}, page_content='Write\n \na\n \nprogram\n \nto\n \ngenerate\n \nAssociation\n \nRules\n \nusing\n \nthe\n \nApriori\n \nalgorithm.\n \n \nTopics\n \nyou\n \nneed\n \nto\n \nlearn\n \n1.\n \nAssociation\n \nRule\n \nMining\n \nIt\n \ndiscovers\n \nrelationships\n \nbetween\n \nitems\n \nin\n \na\n \ndataset.\n \nExample:\n \nCustomers\n \nwho\n \nbuy\n \nbread\n \nmay\n \nalso\n \nbuy\n \nbutter.\n \n \n2.\n \nTransaction\n \nDataset\n \nData\n \nis\n \nstored\n \nas\n \ntransactions\n \ncontaining\n \nitems.\n \nTransactin\n \nItems\n \nT1\n \nBread,\n \nMilk\n \nT2\n \nBread,\n \nButter\n \nT3\n \nBread,

### Embedding and vectorDB Store

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

c:\Users\saige\Documents\claude_test\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(
                f"Model loaded successfully. Embedding dimension: "
                f"{self.model.get_sentence_embedding_dimension()}"
            )
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts

        Args:
            texts: List of text strings to embed

        Returns:
            numpy array of embeddings with shape
            (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")

        embeddings = self.model.encode(
            texts,
            show_progress_bar=True
        )

        print(f"Generated embeddings with shape: {embeddings.shape}")

        return embeddings


# Initialize the embedding manager

embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3949.62it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\saige\AppData\Local\Temp\ipykernel_21008\4090021121.py:22: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  f"{self.model.get_sentence_embedding_dimension()}"


In [8]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(
        self,
        collection_name: str = "pdf_documents",
        persist_directory: str = "../data/vector_store"
    ):
        """
        Initialize the vector store

        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)

            self.client = chromadb.PersistentClient(
                path=self.persist_directory
            )

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )

            print(
                f"Vector store initialized. Collection: "
                f"{self.collection_name}"
            )
            print(
                f"Existing documents in collection: "
                f"{self.collection.count()}"
            )

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(
        self,
        documents: List[Any],
        embeddings: np.ndarray
    ):
        """
        Add documents and their embeddings to the vector store

        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """

        if len(documents) != len(embeddings):
            raise ValueError(
                "Number of documents must match number of embeddings"
            )

        print(f"Adding {len(documents)} documents to vector store...")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(
            zip(documents, embeddings)
        ):

            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())

        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )

            print(
                f"Successfully added {len(documents)} documents "
                f"to vector store"
            )
            print(
                f"Total documents in collection: "
                f"{self.collection.count()}"
            )

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise


# Initialize the vector store

vectorstore = VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [9]:
### Convert text into embeddings
texts=[doc.page_content for doc in chunks]

### generate embeddings for the chunks
embeddings = embedding_manager.generate_embeddings(texts)

### Add the documents and embeddings to the vector store
vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 14 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.25it/s]


Generated embeddings with shape: (14, 384)
Adding 14 documents to vector store...
Successfully added 14 documents to vector store
Total documents in collection: 14


### Retriever pipline from VectorStore

In [10]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(
        self,
        vector_store: VectorStore,
        embedding_manager: EmbeddingManager
    ):
        """
        Initialize the retriever

        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(
        self,
        query: str,
        top_k: int = 5,
        score_threshold: float = 0.0
    ) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query

        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold

        Returns:
            List of dictionaries containing retrieved documents and metadata
        """

        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings(
            [query]
        )[0]

        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            # Process results
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(
                    zip(ids, documents, metadatas, distances)
                ):

                    # Convert distance to similarity score
                    # ChromaDB uses cosine distance
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })

                print(
                    f"Retrieved {len(retrieved_docs)} documents "
                    f"(after filtering)"
                )

            else:
                print("No documents found")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever = RAGRetriever(vectorstore, embedding_manager)

In [11]:
rag_retriever

In [20]:
rag_retriever.retrieve("what is Association Rule Mining ?")

Retrieving documents for query: 'what is Association Rule Mining ?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 59.66it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)


[{'id': 'doc_0fbff80b_0',
  'content': 'Write\n \na\n \nprogram\n \nto\n \ngenerate\n \nAssociation\n \nRules\n \nusing\n \nthe\n \nApriori\n \nalgorithm.\n \n \nTopics\n \nyou\n \nneed\n \nto\n \nlearn\n \n1.\n \nAssociation\n \nRule\n \nMining\n \nIt\n \ndiscovers\n \nrelationships\n \nbetween\n \nitems\n \nin\n \na\n \ndataset.\n \nExample:\n \nCustomers\n \nwho\n \nbuy\n \nbread\n \nmay\n \nalso\n \nbuy\n \nbutter.\n \n \n2.\n \nTransaction\n \nDataset\n \nData\n \nis\n \nstored\n \nas\n \ntransactions\n \ncontaining\n \nitems.\n \nTransactin\n \nItems\n \nT1\n \nBread,\n \nMilk\n \nT2\n \nBread,\n \nButter\n \nT3\n \nBread,\n \nMilk,\n \nButter\n \nT4\n \nMilk,\n \nButter\n \nEach\n \nrow\n \nrepresents\n \none\n \ntransaction.\n \n3.\n \nSupport\n \nSupport\n \nmeasures\n \nhow\n \nfrequently\n \nan\n \nitem\n \nor\n \nitem\n \ncombination\n \noccurs\n \nin\n \nall\n \ntransactions.',
  'metadata': {'content_length': 697,
   'source_file': 'Association Rules using the Apriori alg

### Using Gqok Ai

In [25]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv

load_dotenv()

# 1. Initialize the Groq LLM
groq_api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(
    groq_api_key=groq_api_key,
    model_name="openai/gpt-oss-20b",
    temperature=0.1,
    max_tokens=1024
)


# 2. Simple RAG function: retrieve context + generate response
def rag_simple(query, retriever, llm, top_k=3):

    # Retrieve the context
    results = retriever.retrieve(query, top_k=top_k)

    context = "\n\n".join(
        [doc['content'] for doc in results]
    ) if results else ""

    if not context:
        return "No relevant context found to answer the question."

    # Generate the answer using Groq LLM
    prompt = """Use the following context to answer the question concisely.

    Context:
    {context}

    Question: {query}

    Answer:"""

    response = llm.invoke(
        prompt.format(context=context, query=query)
    )

    return response.content

In [26]:
answer = rag_simple("What is Association Rule Mining?",rag_retriever,llm)
answer

Retrieving documents for query: 'What is Association Rule Mining?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 49.30it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)


'Association Rule Mining is a data mining technique that discovers interesting relationships (rules) between items in a transaction dataset. It identifies frequent itemsets and then generates rules such as “customers who buy bread also buy butter,” based on measures like support and confidence.'

### Using Ollama model

In [27]:
from langchain_ollama import ChatOllama


# 1. Initialize local Ollama LLM
llm_2 = ChatOllama(
    model="llama3.2:latest",
    temperature=0.1,
    num_predict=1024
)


# 2. Simple RAG function: Retrieve context + Generate response
def rag_simple(query, retriever, llm, top_k=3):

    # Retrieve relevant documents
    results = retriever.retrieve(query, top_k=top_k)

    # Combine retrieved document chunks into context
    context = "\n\n".join(
        [doc['content'] for doc in results]
    ) if results else ""

    # Check whether relevant context exists
    if not context:
        return "No relevant context found to answer the question."

    # Create prompt
    prompt = """You are a helpful assistant.

    Answer the question using ONLY the provided context.

    If the answer is not available in the context,
    say: "The answer is not available in the provided document."

    Do not make up information.

    Context:
    {context}

    Question:
    {query}

    Answer:"""

    # Generate answer using local Ollama LLM
    response = llm.invoke(
        prompt.format(context=context, query=query)
    )

    return response.content


# 3. Test the LLM independently
response = llm_2.invoke(
    "Explain Association Rule Mining in simple words."
)

print("LLM Response:")
print(response.content)


# 4. Execute the RAG system
answer = rag_simple(query="What is Association Rule Mining?",retriever=rag_retriever,llm=llm_2)

print("\nRAG Response:")
print(answer)

LLM Response:
Association Rule Mining is a technique used in data analysis to discover patterns in data. Here's a simple explanation:

**What is it?**

Association Rule Mining is a way to find relationships between different items in a dataset. It's like looking for connections between different things.

**How does it work?**

Imagine you have a list of customers and the items they buy. The algorithm looks for patterns in the data, like:

* "If a customer buys A, they are likely to buy B"
* "If a customer buys C, they are likely to buy D"

The algorithm finds these patterns and presents them as rules, like:

* "Buy A and Buy B together"
* "Buy C and Buy D together"

**What's the goal?**

The goal of Association Rule Mining is to identify these patterns and relationships, so businesses can:

* Understand customer behavior
* Identify new sales opportunities
* Improve marketing strategies

For example, a retailer might use Association Rule Mining to discover that customers who buy coffee 

Batches: 100%|██████████| 1/1 [00:00<00:00, 44.41it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)



RAG Response:
Association Rule Mining is the process of discovering relationships between items in a dataset.


### Advance Rag Pipeline

In [28]:
# --- Enhanced RAG Pipeline Features ---

def rag_advanced(
    query,
    retriever,
    llm,
    top_k=5,
    min_score=0.2,
    return_context=False
):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score,
      and optionally full context.
    """

    # Retrieve relevant documents
    results = retriever.retrieve(
        query,
        top_k=top_k,
        score_threshold=min_score
    )

    if not results:
        return {
            'answer': 'No relevant context found.',
            'sources': [],
            'confidence': 0.0,
            'context': ''
        }

    # Prepare context and sources
    context = "\n\n".join(
        [doc['content'] for doc in results]
    )

    sources = [
        {
            'source': doc['metadata'].get(
                'source_file',
                doc['metadata'].get('source', 'unknown')
            ),
            'page': doc['metadata'].get('page', 'unknown'),
            'score': doc['similarity_score'],
            'preview': doc['content'][:300] + '...'
        }
        for doc in results
    ]

    # Calculate confidence score
    confidence = max(
        [doc['similarity_score'] for doc in results]
    )

    # Generate answer
    prompt = """Use the following context to answer the question concisely.

    Context:
    {context}

    Question: {query}

    Answer:"""

    response = llm.invoke(
        prompt.format(context=context, query=query)
    )

    # Prepare output
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }

    # Include full context if requested
    if return_context:
        output['context'] = context

    return output


# Example usage
result = rag_advanced(
    "What is Association Rule Mining?",
    rag_retriever,
    llm,
    top_k=3,
    min_score=0.1,
    return_context=True
)

print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'What is Association Rule Mining?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 22.18it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)


Answer: Association Rule Mining is a data mining technique that discovers interesting relationships (rules) between items in a transaction dataset. It identifies frequent itemsets and then generates rules such as “customers who buy bread also buy butter,” based on measures like support and confidence.
Sources: [{'source': 'Association Rules using the Apriori algorithm.pdf', 'page': 0, 'score': 0.3350032567977905, 'preview': 'Write\n \na\n \nprogram\n \nto\n \ngenerate\n \nAssociation\n \nRules\n \nusing\n \nthe\n \nApriori\n \nalgorithm.\n \n \nTopics\n \nyou\n \nneed\n \nto\n \nlearn\n \n1.\n \nAssociation\n \nRule\n \nMining\n \nIt\n \ndiscovers\n \nrelationships\n \nbetween\n \nitems\n \nin\n \na\n \ndataset.\n \nExample:\n \nCustomers\n \nwho\n \nbuy\n \nbread\n \nmay\n \nalso\n \nbuy\n...'}]
Confidence: 0.3350032567977905
Context Preview: Write
 
a
 
program
 
to
 
generate
 
Association
 
Rules
 
using
 
the
 
Apriori
 
algorithm.
 
 
Topics
 
you
 
need
 
to
 
learn
 
1.
 
Assoc